In [ ]:
%pip install ucimlrepo

In [ ]:
%pip install tabulate

In [ ]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
from ucimlrepo import fetch_ucirepo # Importação de Dados
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from pathlib import Path

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# 1) Análise de Dados

In [ ]:
# Busca do Dataset
rt_iot2022 = fetch_ucirepo(id=942)

# Seperação das features com a coluna Target
x = rt_iot2022.data.features
y = rt_iot2022.data.targets

# União das Linhas e Colunas
df_iot2022 = pd.concat([x, y], axis=1)

# Definição da coluna target
target_column = y.columns[0]

print(f'Shape: {df_iot2022.shape}')
print(f'Target: {target_column}\n')
display(df_iot2022.head())

# 2) Avaliação de Colunas a remover

In [ ]:
constant_columns = [c for c in df_iot2022.columns if df_iot2022[c].nunique() <= 1]
df_iot2022 = df_iot2022.drop(columns=constant_columns, axis=1)

In [ ]:
def cramers_v(var1, var2):
    """
    Lógica de Implementção do V-Cramers para validar a correlação 
    entre as colunas categóricas com a coluna target
    """

    # Criação da tabela de contingência
    main_table = pd.crosstab(var1, var2)

    # Obtenção do Qui-Quadrado (primeiro elemento retornado pela função)
    chi2 = chi2_contingency(main_table, correction=False)[0]

    # Tamanho da Amostra
    n = main_table.sum().sum()

    # Dimensão da Amostra
    r, k = main_table.shape

    # Logica do V Cramer
    phi2 = chi2 / n
    r_corr = max(0, r - (r - 1) ** 2 / (n - 1))
    k_corr = max(0, k - (k - 1) ** 2 / (n - 1))

    phi2corr = max(0, phi2 - ((k - 1) * (r - 1)) / (n - 1))
    min_corr = min(k_corr - 1, r_corr - 1)

    return np.sqrt(phi2corr/min_corr) if min_corr != 0 else 0

In [ ]:
# Testagem com novas features
df_test = df_iot2022.copy()

df_test['is_same_port'] = (df_iot2022['id.orig_p'] == df_iot2022['id.resp_p']).astype("int")
df_test['both_known_well'] = ((df_iot2022['id.orig_p'] < 1024) & (df_iot2022['id.resp_p'] < 1024)).astype("int") # server to server
df_test['orig_is_dinamic'] = (df_iot2022['id.orig_p'] >= 32768).astype("int")
df_test['resp_known_well'] = (df_iot2022['id.resp_p'] < 1024).astype("int")

results = []
for c in ['id.orig_p', 'id.resp_p', 'is_same_port', 'both_known_well', 'orig_is_dinamic', 'resp_known_well']:
    v = cramers_v(df_test[c], df_test[target_column])
    mi = mutual_info_classif(df_test[[c]], df_test[target_column], discrete_features=True, random_state=42)[0]
    results.append({"coluna": c, "Cramér's V": round(v, 4), "Mutual Info": round(mi, 4)})

print(pd.DataFrame(results).to_markdown(index=False))

# 3) Split Estratificado

In [ ]:
X_train, X_test, Y_train, Y_test = train